# Chapter 19 Companion Notebook: Case Studies and Capstone Projects

This notebook reproduces Chapter 19's end-to-end case study on the UCI/Kaggle "Default of Credit Card Clients" dataset (Yeh and Lien, 2009), downloading the real dataset directly from the UCI Machine Learning Repository, including an isotonic recalibration of the plain logistic regression (Section 6b) that fixes its lowest-decile miscalibration without retraining or discarding the model.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Download and load the raw dataset

In [1]:
import os
import urllib.request
import numpy as np
import pandas as pd

DATA_URL = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
DATA_DIR = "ch16_data"
ZIP_PATH = os.path.join(DATA_DIR, "uci_credit.zip")
XLS_PATH = os.path.join(DATA_DIR, "default of credit card clients.xls")

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(XLS_PATH):
    import zipfile
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

df = pd.read_excel(XLS_PATH, header=1)
df = df.rename(columns={"default payment next month": "default", "PAY_0": "PAY_1"})
print("shape:", df.shape)
print("default rate:", df['default'].mean())
df[['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']].describe().round(1)

shape: (30000, 25)
default rate: 0.2212


,LIMIT_BAL,AGE,BILL_AMT1,PAY_AMT1
count,30000.0,30000.0,30000.0,30000.0
mean,167484.3,35.5,51223.3,5663.6
std,129747.7,9.2,73635.9,16563.3
min,10000.0,21.0,-165580.0,0.0
25%,50000.0,28.0,3558.8,1000.0
50%,140000.0,34.0,22381.5,2100.0
75%,240000.0,41.0,67091.0,5006.0
max,1000000.0,79.0,964511.0,873552.0


## 2. Clean undocumented category codes

In [2]:
print("EDUCATION before cleaning:")
print(df['EDUCATION'].value_counts().sort_index())
print("\nMARRIAGE before cleaning:")
print(df['MARRIAGE'].value_counts().sort_index())

df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print("\nEDUCATION after cleaning:")
print(df['EDUCATION'].value_counts().sort_index())

EDUCATION before cleaning:
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

MARRIAGE before cleaning:
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

EDUCATION after cleaning:
EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64


## 3. Feature engineering

In [3]:
pay_cols = [f"PAY_{i}" for i in [1, 2, 3, 4, 5, 6]]
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
paid_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

df['max_delinquency'] = df[pay_cols].max(axis=1)
df['avg_utilization'] = (df[bill_cols].div(df['LIMIT_BAL'], axis=0)).mean(axis=1)
df['total_bill'] = df[bill_cols].sum(axis=1)
df['total_paid'] = df[paid_cols].sum(axis=1)
df['pay_to_bill_ratio'] = (df['total_paid'] / df['total_bill'].replace(0, np.nan)).fillna(0).clip(upper=5)

feature_cols = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE'] + pay_cols + bill_cols + paid_cols + \
    ['max_delinquency', 'avg_utilization', 'total_bill', 'total_paid', 'pay_to_bill_ratio']
print("number of features:", len(feature_cols))

number of features: 28


## 4. Stratified train/test split

In [4]:
from sklearn.model_selection import train_test_split

X = df[feature_cols].copy()
y = df['default'].copy()
sex = df['SEX'].copy()

X_train, X_test, y_train, y_test, sex_train, sex_test = train_test_split(
    X, y, sex, test_size=0.2, stratify=y, random_state=42)
print("train shape:", X_train.shape, " test shape:", X_test.shape)
print("train default rate:", y_train.mean(), " test default rate:", y_test.mean())

train shape: (24000, 28)  test shape: (6000, 28)
train default rate: 0.22120833333333334  test default rate: 0.22116666666666668


## 5. Model bake-off: logistic regression, random forest, XGBoost

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, brier_score_loss)
import xgboost as xgb

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {}
lr_plain = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)
models['Logistic (plain)'] = (lr_plain, X_test_s)

lr_bal = LogisticRegression(max_iter=2000, class_weight='balanced').fit(X_train_s, y_train)
models['Logistic (balanced)'] = (lr_bal, X_test_s)

rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(X_train, y_train)
models['Random Forest'] = (rf, X_test)

xgb_model = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                               colsample_bytree=0.8, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
models['XGBoost'] = (xgb_model, X_test)

rows = []
for name, (model, Xte) in models.items():
    proba = model.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, proba),
        'pr_auc': average_precision_score(y_test, proba),
        'brier': brier_score_loss(y_test, proba),
    })
pd.DataFrame(rows).set_index('model').round(4)

,accuracy,precision,recall,f1,roc_auc,pr_auc,brier
model,,,,,,,
Logistic (plain),0.8082,0.6964,0.2351,0.3515,0.7244,0.4987,0.1455
Logistic (balanced),0.7198,0.4127,0.6307,0.4990,0.7250,0.4892,0.2049
Random Forest,0.8168,0.6624,0.3504,0.4584,0.7748,0.5554,0.1360
XGBoost,0.8180,0.6648,0.3572,0.4647,0.7795,0.5576,0.1349


## 6. Calibration: reliability by decile

In [6]:
for name in ['Logistic (plain)', 'XGBoost']:
    model, Xte = models[name]
    proba = model.predict_proba(Xte)[:, 1]
    bins = pd.qcut(proba, 10, duplicates='drop')
    cal = pd.DataFrame({'proba': proba, 'actual': y_test.values, 'bin': bins})
    grouped = cal.groupby('bin', observed=True).agg(mean_pred=('proba', 'mean'), mean_actual=('actual', 'mean'), n=('actual', 'size'))
    print(f"--- {name} ---")
    print(grouped.round(4))
    print()

--- Logistic (plain) ---
                         mean_pred  mean_actual    n
bin                                                 
(-0.0009999815, 0.0573]     0.0400       0.1250  600
(0.0573, 0.0963]            0.0790       0.1217  600
(0.0963, 0.124]             0.1101       0.1350  600
(0.124, 0.152]              0.1383       0.1067  600
(0.152, 0.173]              0.1630       0.0983  600
(0.173, 0.198]              0.1847       0.1400  600
(0.198, 0.249]              0.2171       0.1617  600
(0.249, 0.334]              0.2931       0.2500  600
(0.334, 0.469]              0.4025       0.4167  600
(0.469, 0.992]              0.5809       0.6567  600

--- XGBoost ---
                                mean_pred  mean_actual    n
bin                                                        
(0.012199999999999999, 0.0502]     0.0359       0.0383  600
(0.0502, 0.0728]                   0.0611       0.0733  600
(0.0728, 0.0956]                   0.0843       0.0933  600
(0.0956, 0.119]       

### 6b. Fixing miscalibration: isotonic recalibration

Recalibrate the plain logistic regression using 5-fold cross-validated isotonic regression (fit within the training set only), and check whether the Brier score and lowest-bin calibration gap improve.

In [7]:
from sklearn.calibration import CalibratedClassifierCV

proba_plain = lr_plain.predict_proba(X_test_s)[:, 1]
brier_plain = brier_score_loss(y_test, proba_plain)

calib_lr = CalibratedClassifierCV(LogisticRegression(max_iter=2000), method='isotonic', cv=5)
calib_lr.fit(X_train_s, y_train)
proba_calib = calib_lr.predict_proba(X_test_s)[:, 1]
brier_calib = brier_score_loss(y_test, proba_calib)

print(f"Plain logistic Brier: {brier_plain:.4f}")
print(f"Isotonic-recalibrated Brier: {brier_calib:.4f}")
print(f"ROC-AUC plain: {roc_auc_score(y_test, proba_plain):.4f}")
print(f"ROC-AUC recalibrated: {roc_auc_score(y_test, proba_calib):.4f}")

bins_calib = pd.qcut(proba_calib, 10, duplicates='drop')
cal_calib = pd.DataFrame({'proba': proba_calib, 'actual': y_test.values, 'bin': bins_calib})
grouped_calib = cal_calib.groupby('bin', observed=True).agg(
    mean_pred=('proba', 'mean'), mean_actual=('actual', 'mean'), n=('actual', 'size'))
print("\nRecalibrated reliability table:")
print(grouped_calib.round(4))

Plain logistic Brier: 0.1455
Isotonic-recalibrated Brier: 0.1419
ROC-AUC plain: 0.7244
ROC-AUC recalibrated: 0.7265

Recalibrated reliability table:
                               mean_pred  mean_actual     n
bin                                                        
(0.007330000000000001, 0.105]     0.0988       0.1186   885
(0.105, 0.109]                    0.1089       0.1376   356
(0.109, 0.111]                    0.1105       0.1137  1645
(0.111, 0.113]                    0.1125       0.0921   152
(0.113, 0.149]                    0.1324       0.1338   613
(0.149, 0.213]                    0.1743       0.1787   554
(0.213, 0.305]                    0.2608       0.2466   596
(0.305, 0.586]                    0.4376       0.4126   601
(0.586, 0.913]                    0.6772       0.6622   598


## 7. Global explanations: permutation importance and SHAP

In [8]:
from sklearn.inspection import permutation_importance
import shap

perm = permutation_importance(xgb_model, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc', n_jobs=-1)
imp_df = pd.DataFrame({'feature': feature_cols, 'importance_mean': perm.importances_mean}).sort_values(
    'importance_mean', ascending=False)
print("Top 10 by permutation importance:")
print(imp_df.head(10).to_string(index=False))

explainer = shap.TreeExplainer(xgb_model)
X_test_sample = X_test.iloc[:1000]
shap_values = explainer(X_test_sample)
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_imp = pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': mean_abs_shap}).sort_values(
    'mean_abs_shap', ascending=False)
print("\nTop 10 by mean |SHAP|:")
print(shap_imp.head(10).to_string(index=False))

Top 10 by permutation importance:
          feature  importance_mean
            PAY_1         0.034348
  max_delinquency         0.030012
        BILL_AMT1         0.011881
  avg_utilization         0.010306
        LIMIT_BAL         0.006055
       total_paid         0.004718
         PAY_AMT2         0.004325
       total_bill         0.003343
        BILL_AMT5         0.003299
pay_to_bill_ratio         0.003219

Top 10 by mean |SHAP|:
          feature  mean_abs_shap
  max_delinquency       0.441699
            PAY_1       0.341088
  avg_utilization       0.150125
        LIMIT_BAL       0.139522
        BILL_AMT1       0.133602
         PAY_AMT2       0.090235
       total_paid       0.077592
pay_to_bill_ratio       0.067078
         PAY_AMT3       0.066885
         PAY_AMT1       0.066581


## 8. Individual explanations: a low-risk and a high-risk applicant

In [9]:
idx_low = 0
applicant_low = X_test_sample.iloc[idx_low]
proba_low = xgb_model.predict_proba(X_test_sample.iloc[[idx_low]])[0, 1]
top_low = pd.DataFrame({'feature': feature_cols, 'value': applicant_low.values,
                        'shap': shap_values.values[idx_low]}).sort_values('shap', key=abs, ascending=False)
print("Low-risk applicant, predicted default probability:", round(proba_low, 4))
print(top_low.head(6).to_string(index=False))
print("base value:", explainer.expected_value, " sum shap + base:",
      shap_values.values[idx_low].sum() + explainer.expected_value)

proba_sample = xgb_model.predict_proba(X_test_sample)[:, 1]
idx_high = int(np.argmax(proba_sample))
applicant_high = X_test_sample.iloc[idx_high]
top_high = pd.DataFrame({'feature': feature_cols, 'value': applicant_high.values,
                         'shap': shap_values.values[idx_high]}).sort_values('shap', key=abs, ascending=False)
print("\nHigh-risk applicant, predicted default probability:", round(proba_sample[idx_high], 4))
print(top_high.head(6).to_string(index=False))
print("base value:", explainer.expected_value, " sum shap + base:",
      shap_values.values[idx_high].sum() + explainer.expected_value)

Low-risk applicant, predicted default probability: 0.1197
          feature        value      shap
  max_delinquency     0.000000 -0.319925
  avg_utilization     0.121113 -0.164454
        LIMIT_BAL 50000.000000  0.160512
            PAY_1    -1.000000 -0.159547
pay_to_bill_ratio     1.003385 -0.120675
        BILL_AMT3     0.000000 -0.106728
base value: -1.2716084  sum shap + base: -1.9949336

High-risk applicant, predicted default probability: 0.9116
          feature   value     shap
            PAY_1     3.0 1.091443
  max_delinquency     7.0 0.896956
        BILL_AMT1  2400.0 0.181129
            PAY_6     7.0 0.178795
        LIMIT_BAL 20000.0 0.159088
pay_to_bill_ratio     0.0 0.146245
base value: -1.2716084  sum shap + base: 2.3328946


### A worked example: from predicted probability to a provisioning number (EL = PD x LGD x EAD)

In [10]:
LGD = 0.60  # Chapter 9's illustrative loss-given-default assumption, reused here for an unsecured card balance

ead_low = applicant_low["LIMIT_BAL"]
ead_high = applicant_high["LIMIT_BAL"]

el_low = proba_low * LGD * ead_low
el_high = proba_sample[idx_high] * LGD * ead_high

print(f"Low-risk applicant:  PD={proba_low:.4f}  EAD=${ead_low:,.0f}  LGD={LGD}  ->  EL=${el_low:,.2f}")
print(f"High-risk applicant: PD={proba_sample[idx_high]:.4f}  EAD=${ead_high:,.0f}  LGD={LGD}  ->  EL=${el_high:,.2f}")
print(f"\nEL ratio (high/low): {el_high/el_low:.2f}")
print(f"PD ratio (high/low): {proba_sample[idx_high]/proba_low:.2f}")

Low-risk applicant:  PD=0.1197  EAD=$50,000  LGD=0.6  ->  EL=$3,592.08
High-risk applicant: PD=0.9116  EAD=$20,000  LGD=0.6  ->  EL=$10,938.78

EL ratio (high/low): 3.05
PD ratio (high/low): 7.61


## 9. Cost-sensitive threshold selection

In [11]:
from sklearn.metrics import confusion_matrix

proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
cost_fn, cost_fp = 500, 50
thresholds = np.arange(0.05, 0.61, 0.01)
cost_table = []
for t in thresholds:
    pred_t = (proba_xgb >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    cost_table.append((t, fn, fp, fn * cost_fn + fp * cost_fp))
cost_df = pd.DataFrame(cost_table, columns=['threshold', 'fn', 'fp', 'total_cost'])
best_row = cost_df.loc[cost_df['total_cost'].idxmin()]
print(cost_df.iloc[::10].to_string(index=False))
print("\ncost-minimizing threshold:", best_row['threshold'], " total cost:", best_row['total_cost'])
print("cost at naive 0.5 threshold:", cost_df.loc[np.isclose(cost_df['threshold'], 0.5), 'total_cost'].values[0])
best_t = best_row['threshold']

 threshold  fn   fp  total_cost
      0.05  23 4107      216850
      0.15 293 1873      240150
      0.25 515  856      300300
      0.35 671  508      360900
      0.45 802  291      415550
      0.55 884  211      452550

cost-minimizing threshold: 0.09000000000000001  total cost: 211100.0
cost at naive 0.5 threshold: 438450


## 10. Fairness audit by sex

In [12]:
pred_final = (proba_xgb >= best_t).astype(int)
audit_df = pd.DataFrame({'sex': sex_test.values, 'actual': y_test.values, 'pred_default': pred_final, 'proba': proba_xgb})

results = {}
for s, label in [(1, 'Male'), (2, 'Female')]:
    sub = audit_df[audit_df['sex'] == s]
    approval_rate = (sub['pred_default'] == 0).mean()
    defaulters = sub[sub['actual'] == 1]
    fnr = defaulters['pred_default'].eq(0).mean()
    non_defaulters = sub[sub['actual'] == 0]
    fpr = non_defaulters['pred_default'].eq(1).mean()
    results[label] = approval_rate
    print(f"{label}: n={len(sub)} approval_rate={approval_rate:.4f} FNR={fnr:.4f} FPR={fpr:.4f}")

air = min(results.values()) / max(results.values())
print("\nadverse impact ratio:", round(air, 4))

Male: n=2402 approval_rate=0.2410 FNR=0.0642 FPR=0.7051
Female: n=3598 approval_rate=0.2946 FNR=0.0940 FPR=0.6511

adverse impact ratio: 0.8182


## 11. Hyperparameter tuning via cross-validation

In [13]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_dist = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [100, 200, 300, 500],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}
base_xgb = xgb.XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(base_xgb, param_distributions=param_dist, n_iter=30, scoring='roc_auc',
                             cv=cv, random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
print("best params:", search.best_params_)
print("best CV ROC-AUC:", round(search.best_score_, 4))

tuned_model = search.best_estimator_
proba_tuned = tuned_model.predict_proba(X_test)[:, 1]
print("tuned test ROC-AUC:", round(roc_auc_score(y_test, proba_tuned), 4))
print("tuned test PR-AUC:", round(average_precision_score(y_test, proba_tuned), 4))
print("tuned test Brier:", round(brier_score_loss(y_test, proba_tuned), 4))
print("untuned test ROC-AUC (for comparison):", round(roc_auc_score(y_test, proba_xgb), 4))

best params: {'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
best CV ROC-AUC: 0.7863
tuned test ROC-AUC: 0.7802
tuned test PR-AUC: 0.5625
tuned test Brier: 0.1345
untuned test ROC-AUC (for comparison): 0.7795


## 12. Intersectional fairness check: sex crossed with age group

In [14]:
age_test = X_test['AGE'].values
audit_df['age_group'] = np.where(age_test < 30, 'Under 30', '30 and over')

intersect_results = {}
for s, slabel in [(1, 'Male'), (2, 'Female')]:
    for ag in ['Under 30', '30 and over']:
        sub = audit_df[(audit_df['sex'] == s) & (audit_df['age_group'] == ag)]
        ar = (sub['pred_default'] == 0).mean()
        defaulters = sub[sub['actual'] == 1]
        fnr = defaulters['pred_default'].eq(0).mean() if len(defaulters) > 0 else float('nan')
        intersect_results[(slabel, ag)] = ar
        print(f"{slabel}, {ag}: n={len(sub)} approval_rate={ar:.4f} FNR={fnr:.4f}")

air_intersect = min(intersect_results.values()) / max(intersect_results.values())
print("\nintersectional adverse impact ratio:", round(air_intersect, 4))

Male, Under 30: n=669 approval_rate=0.1988 FNR=0.0200
Male, 30 and over: n=1733 approval_rate=0.2574 FNR=0.0803
Female, Under 30: n=1243 approval_rate=0.2864 FNR=0.0903
Female, 30 and over: n=2355 approval_rate=0.2989 FNR=0.0961

intersectional adverse impact ratio: 0.665
